In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import re
import string
import os
from tqdm.notebook import tqdm

# Cek ketersediaan GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# Konfigurasi Path Data
DATA_PATH = "/kaggle/input/nlp-getting-started/train.csv"

# --- Hyperparameter GAN ---
NOISE_DIM = 100       # Dimensi vektor noise input untuk Generator
EMBED_DIM = 128       # Dimensi embedding kata
HIDDEN_DIM = 256      # Ukuran hidden layer di G dan D
LAMBDA_GP = 10        # Koefisien untuk Gradient Penalty (standar WGAN-GP)
N_CRITIC = 5          # Jumlah D dilatih per 1 G dilatih (standar WGAN-GP)

# --- Hyperparameter Data & Training ---
TARGET_CLASS = 1      # 1 untuk "Disaster", 0 untuk "Not Disaster"
MAX_LENGTH = 30       # Panjang maksimum tweet (dipadatkan/dipotong)
MIN_COUNT = 2         # Frekuensi kata minimum untuk vocab
BATCH_SIZE = 64
EPOCHS = 50           # GAN perlu waktu, 50-100+ epoch adalah normal
LR = 0.0002           # Learning rate rendah umum untuk GAN
B1 = 0.5              # Beta1 untuk Adam optimizer
B2 = 0.999            # Beta2 untuk Adam optimizer

# Token khusus
PAD_token = 0  # Padding
SOS_token = 1  # Start-of-Sentence
EOS_token = 2  # End-of-Sentence
UNK_token = 3  # Unknown word

In [ ]:
class Voc:
    """Kelas untuk mengelola Vocabulary"""
    def __init__(self, name):
        self.name = name
        self.trimmed = False
        self.word2index = {"PAD": PAD_token, "SOS": SOS_token, "EOS": EOS_token, "UNK": UNK_token}
        self.word2count = {}
        self.index2word = {PAD_token: "PAD", SOS_token: "SOS", EOS_token: "EOS", UNK_token: "UNK"}
        self.num_words = 4

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.num_words
            self.word2count[word] = 1
            self.index2word[self.num_words] = word
            self.num_words += 1
        else:
            self.word2count[word] += 1

    def trim(self, min_count):
        if self.trimmed:
            return
        self.trimmed = True
        keep_words = []
        for k, v in self.word2count.items():
            if v >= min_count:
                keep_words.append(k)

        print(f'keep_words {len(keep_words)} / {len(self.word2index)-4} = {len(keep_words)/(len(self.word2index)-4):.4f}')
        self.word2index = {"PAD": PAD_token, "SOS": SOS_token, "EOS": EOS_token, "UNK": UNK_token}
        self.word2count = {}
        self.index2word = {PAD_token: "PAD", SOS_token: "SOS", EOS_token: "EOS", UNK_token: "UNK"}
        self.num_words = 4
        for word in keep_words:
            self.addWord(word)

# Normalisasi Tweet
def normalizeString(s):
    s = s.lower().strip()
    # Hapus URL
    s = re.sub(r'https?://\S+|www\.\S+', '', s)
    # Hapus handle twitter (@) dan hashtag (#)
    s = re.sub(r'[@#]\w+', '', s)
    # Hapus tanda baca
    s = s.translate(str.maketrans('', '', string.punctuation))
    # Hapus angka
    s = re.sub(r'\d+', '', s)
    # Hapus spasi berlebih
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def loadData(path, target_class):
    print("Membaca data...")
    df = pd.read_csv(path)
    # Filter hanya untuk kelas yang kita inginkan
    df_filtered = df[df['target'] == target_class]
    
    print(f"Menggunakan {len(df_filtered)} sampel dari kelas {target_class}")
    
    voc = Voc("DisasterTweets")
    sentences = []
    for s in df_filtered['text']:
        norm_s = normalizeString(s)
        if norm_s:
            voc.addSentence(norm_s)
            sentences.append(norm_s)
            
    print("Membangun Vocab...")
    voc.trim(MIN_COUNT)
    
    # Filter kalimat yang mungkin katanya ter-trim
    final_sentences = []
    for s in sentences:
        words = s.split(' ')
        keep = True
        for w in words:
            if w not in voc.word2index:
                keep = False
                break
        if keep and len(words) > 0:
            final_sentences.append(s)
            
    print(f"Total Vocab Size: {voc.num_words}")
    print(f"Total kalimat final: {len(final_sentences)}")
    return voc, final_sentences

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, sentences, voc, max_len):
        self.sentences = sentences
        self.voc = voc
        self.max_len = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        words = sentence.split(' ')
        
        # Konversi ke index
        indices = [self.voc.word2index.get(w, UNK_token) for w in words]
        indices.append(EOS_token) # Tambah EOS
        
        # Padding / Truncation
        if len(indices) < self.max_len:
            indices += [PAD_token] * (self.max_len - len(indices))
        else:
            indices = indices[:self.max_len]
            if indices[-1] != EOS_token:
                indices[-1] = EOS_token # Pastikan diakhiri EOS

        return torch.LongTensor(indices)

# Muat data dan buat DataLoader
try:
    voc, sentences = loadData(DATA_PATH, TARGET_CLASS)
    
    dataset = TweetDataset(sentences, voc, MAX_LENGTH)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    
    print("\nDataLoader siap.")
    batch_ex = next(iter(dataloader))
    print(f"Contoh shape batch: {batch_ex.shape}")
    
except FileNotFoundError:
    print("="*50)
    print("ERROR: DATASET TIDAK DITEMUKAN")
    print("Pastikan Anda telah menambahkan 'NLP with Disaster Tweets' ke Kaggle.")
    print("="*50)
    dataloader = None

In [ ]:
class Generator(nn.Module):
    def __init__(self, noise_dim, embed_dim, hidden_dim, max_len):
        super(Generator, self).__init__()
        self.max_len = max_len
        self.hidden_dim = hidden_dim
        
        # Layer untuk memproses noise awal -> hidden state GRU
        self.noise_to_hidden = nn.Sequential(
            nn.Linear(noise_dim, hidden_dim),
            nn.LeakyReLU(0.2)
        )
        
        # GRU
        # Inputnya adalah real_embeddings (embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=1, batch_first=True)
        
        # Output layer
        self.out = nn.Sequential(
            nn.Linear(hidden_dim, embed_dim),
            nn.Tanh() 
        )

    def forward(self, real_embeddings, noise):
        # real_embeddings shape: (batch_size, max_len, embed_dim)
        # noise shape: (batch_size, noise_dim)
        batch_size = noise.size(0)
        
        # 1. Buat hidden state awal dari noise
        # h0 shape: (1, batch_size, hidden_dim)
        h0 = self.noise_to_hidden(noise).unsqueeze(0) 
        
        # 2. Jalankan GRU
        # Inputnya adalah real_embeddings, hidden state awalnya dari noise
        # gru_out shape: (batch_size, max_len, hidden_dim)
        gru_out, _ = self.gru(real_embeddings, h0)
        
        # 3. Proyeksikan ke ruang embedding
        # output shape: (batch_size, max_len, embed_dim)
        # Kita tambahkan (residual) dengan input asli agar G hanya belajar "mengedit"
        output = self.out(gru_out) + real_embeddings
        
        return output

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, embed_dim, hidden_dim):
        super(Discriminator, self).__init__()
        
        # GRU untuk memproses sekuens embedding
        # Kita akan menggunakan GRU yang sama untuk kedua input
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=1, 
                          batch_first=True, bidirectional=True)
        
        # Classifier head
        # Inputnya adalah gabungan dari 2 hidden state (asli + augmentasi)
        # Masing-masing (hidden_dim * 2), jadi totalnya (hidden_dim * 2 * 2)
        self.out = nn.Sequential(
            nn.Linear(hidden_dim * 2 * 2, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1)
            # TIDAK ADA SIGMOID! (WGAN)
        )

    def _get_hidden_state(self, embeddings):
        # embeddings shape: (batch_size, max_len, embed_dim)
        
        # _, h_n shape: (num_layers*num_dir, batch_size, hidden_dim)
        _, h_n = self.gru(embeddings)
        
        # Ambil hidden state terakhir dari kedua arah
        # h_n shape: (2, batch_size, hidden_dim) -> (batch_size, hidden_dim * 2)
        hidden = torch.cat((h_n[0], h_n[1]), dim=1)
        return hidden

    def forward(self, condition_embeddings, generated_embeddings):
        # Dapatkan hidden state dari teks kondisi (asli)
        h_condition = self._get_hidden_state(condition_embeddings)
        
        # Dapatkan hidden state dari teks yang di-generate
        h_generated = self._get_hidden_state(generated_embeddings)
        
        # Gabungkan keduanya
        h_concat = torch.cat((h_condition, h_generated), dim=1)
        
        # Dapatkan skor (kritik)
        score = self.out(h_concat)
        return score

In [ ]:
def weights_init(m):
    # Inisialisasi bobot (membantu stabilitas GAN)
    classname = m.__class__.__name__
    if classname.find('Linear') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0)
    elif classname.find('GRU') != -1:
        for param in m.parameters():
            if len(param.shape) >= 2:
                nn.init.orthogonal_(param.data)
            else:
                nn.init.normal_(param.data)

def compute_gradient_penalty(D, real_condition, real_samples, fake_samples):
    """Menghitung gradient penalty untuk WGAN-GP (Conditional)"""
    # real_condition: Teks asli (Kondisi)
    # real_samples: Teks asli (sebagai augmentasi "sempurna")
    # fake_samples: Teks augmentasi (dari G)
    
    # Sampel acak (alpha)
    alpha = torch.rand(real_samples.size(0), 1, 1, device=device)
    alpha = alpha.expand_as(real_samples)
    
    # Buat interpolasi antara sampel augmentasi asli dan palsu
    interpolated = (alpha * real_samples + (1 - alpha) * fake_samples).requires_grad_(True)
    
    # Matikan CuDNN untuk double backward (Perbaikan dari error sebelumnya)
    with torch.backends.cudnn.flags(enabled=False):
        d_interpolated = D(real_condition, interpolated)
    
    # Hitung gradien
    grad_outputs = torch.ones(d_interpolated.size(), device=device, requires_grad=False)
    gradients = torch.autograd.grad(
        outputs=d_interpolated,
        inputs=interpolated,  # Gradien dihitung thd 'interpolated'
        grad_outputs=grad_outputs,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    
    gradients = gradients.reshape(gradients.size(0), -1) # Perbaikan dari error .view()
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty

In [ ]:
if dataloader is not None:
    # --- Inisialisasi ---
    
    # 1. Layer Embedding
    embedding_layer = nn.Embedding(voc.num_words, EMBED_DIM).to(device)
    nn.init.normal_(embedding_layer.weight.data, 0.0, 0.02)
    
    # 2. Models (Gunakan G dan D dari Sel 6 & 7 baru)
    generator = Generator(NOISE_DIM, EMBED_DIM, HIDDEN_DIM, MAX_LENGTH).to(device)
    discriminator = Discriminator(EMBED_DIM, HIDDEN_DIM).to(device)
    
    generator.apply(weights_init)
    discriminator.apply(weights_init)

    # 3. Optimizers
    # **PERBAIKAN STABILITAS:** Buat D belajar 10x lebih lambat dari G
    optimizer_G = optim.Adam(
        list(generator.parameters()) + list(embedding_layer.parameters()),
        lr=LR, betas=(B1, B2)
    )
    optimizer_D = optim.Adam(
        list(discriminator.parameters()) + list(embedding_layer.parameters()),
        lr=LR * 0.1, betas=(B1, B2) # <-- D LEBIH LAMBAT
    )
    
    print("Memulai Training cWGAN-GP...")

    # --- Training Loop ---
    for epoch in range(1, EPOCHS + 1):
        for i, real_indices in enumerate(tqdm(dataloader, desc=f"Epoch {epoch}/{EPOCHS}")):
            
            real_indices = real_indices.to(device)
            batch_size = real_indices.size(0)
            
            # Ubah data asli menjadi embeddings (ini adalah 'kondisi' kita)
            real_embeddings = embedding_layer(real_indices)

            # =================================
            #  Latih Discriminator (Critic)
            # =================================
            for _ in range(N_CRITIC):
                discriminator.zero_grad()
                
                # 1. Dapatkan skor untuk data Asli
                # Inputnya adalah (Kondisi=Asli, Augmentasi=Asli)
                d_real = discriminator(real_embeddings, real_embeddings)
                
                # 2. Dapatkan skor untuk data Palsu
                noise = torch.randn(batch_size, NOISE_DIM, device=device)
                fake_embeddings = generator(real_embeddings, noise)
                
                # Inputnya adalah (Kondisi=Asli, Augmentasi=Palsu)
                d_fake = discriminator(real_embeddings, fake_embeddings.detach()) # detach()
                
                # 3. Hitung Gradient Penalty
                gp = compute_gradient_penalty(discriminator, 
                                              real_embeddings.data,    # Kondisi
                                              real_embeddings.data,    # Augmentasi Asli
                                              fake_embeddings.data)    # Augmentasi Palsu
                
                # 4. Hitung Loss D
                d_loss = torch.mean(d_fake) - torch.mean(d_real) + LAMBDA_GP * gp
                
                # Backward pass
                d_loss.backward(retain_graph=True)
                optimizer_D.step()

            # =================================
            #  Latih Generator
            # =================================
            generator.zero_grad()
            
            # Buat data palsu baru
            noise = torch.randn(batch_size, NOISE_DIM, device=device)
            # Dapatkan embedding palsu (conditional)
            fake_embeddings = generator(real_embeddings, noise)
            
            # Dapatkan skor D untuk data palsu
            # Inputnya (Kondisi=Asli, Augmentasi=Palsu)
            # G ingin loss-nya (yaitu -skor) sekecil mungkin
            g_loss = -torch.mean(discriminator(real_embeddings, fake_embeddings))
            
            # Backward pass
            g_loss.backward()
            optimizer_G.step()
        
        # --- Akhir Epoch ---
        print(f"[Epoch {epoch}/{EPOCHS}] [D loss: {d_loss.item():.4f}] [G loss: {g_loss.item():.4f}]")
        
        # Simpan output epoch 10 untuk perbandingan
        if epoch == 10:
            print("\n--- Menyimpan output Epoch 10 ---")
            generator.eval()
            with torch.no_grad():
                noise_test = torch.randn(batch_size, NOISE_DIM, device=device)
                # Ambil batch pertama lagi untuk konsistensi
                real_indices_test = next(iter(dataloader)).to(device)
                real_embed_test = embedding_layer(real_indices_test)
                fake_embed_test_10 = generator(real_embed_test, noise_test)
            generator.train()


    print("Pelatihan Selesai!")
else:
    print("Training dibatalkan karena data tidak dimuat.")

In [ ]:
import torch.nn.functional as F
def decode_embeddings(embeddings, embedding_matrix, voc):
    """
    Mengubah batch embedding palsu kembali menjadi teks.
    
    Args:
        embeddings (Tensor): (batch_size, max_len, embed_dim)
        embedding_matrix (Tensor): (vocab_size, embed_dim)
        voc (Voc): Objek vocabulary
    """
    generated_texts = []
    
    # Normalisasi embedding matrix untuk pencarian (Cosine Similarity)
    # Ini opsional tapi seringkali lebih baik dari L2 distance
    embedding_matrix_norm = F.normalize(embedding_matrix, p=2, dim=1)
    
    for i in range(embeddings.size(0)): # Loop per kalimat di batch
        seq_text = []
        seq_embeds = embeddings[i] # (max_len, embed_dim)
        
        # Normalisasi embedding palsu
        seq_embeds_norm = F.normalize(seq_embeds, p=2, dim=1)
        
        # Hitung cosine similarity (dot product dari vektor ternormalisasi)
        # sim shape: (max_len, vocab_size)
        sim = torch.matmul(seq_embeds_norm, embedding_matrix_norm.t())
        
        # Ambil indeks kata dengan similarity tertinggi
        # indices shape: (max_len)
        _, indices = torch.max(sim, dim=1)
        
        for idx in indices:
            word = voc.index2word[idx.item()]
            if word == "EOS":
                break
            if word != "PAD" and word != "SOS" and word != "UNK":
                seq_text.append(word)
        
        generated_texts.append(" ".join(seq_text))
        
    return generated_texts

# --- Jalankan Augmentasi ---
if 'generator' in locals():
    print("="*50)
    print("Contoh Teks Augmentasi (Conditional GAN):")
    print("="*50)
    
    generator.eval()
    
    N_TO_SHOW = 5 
    
    with torch.no_grad():
        # ... (kode untuk ambil embedding_weights dan real_indices_test) ...
            
        real_embeddings_test = embedding_layer(real_indices_test)
        
        # Buat noise
        noise = torch.randn(real_embeddings_test.size(0), NOISE_DIM, device=device)
        
        # Hasilkan embedding palsu (conditional)
        fake_embeddings_test = generator(real_embeddings_test, noise)
        
        # Decode kedua versi
        original_tweets = decode_embeddings(real_embeddings_test, embedding_weights, voc)
        generated_tweets = decode_embeddings(fake_embeddings_test, embedding_weights, voc)
        
        for i, tweet in enumerate(generated_tweets):
            print(f"[{i+1}] {tweet}")
            
    generator.train()
else:
    print("Model tidak dilatih.")

# **Testing with Classifier**

In [ ]:
!pip install transformers datasets --quiet

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report

from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from torch.optim import AdamW

# Pastikan Pustaka PyTorch/Pandas/Tqdm sudah ada dari sel sebelumnya
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
import time

In [ ]:
# Model classifier yang akan kita gunakan (cepat dan bagus)
CLASSIFIER_MODEL_NAME = "distilbert-base-uncased" 

# Hyperparameter untuk classifier
CLASSIFIER_EPOCHS = 3       # 3 epoch biasanya cukup untuk fine-tuning
CLASSIFIER_BATCH_SIZE = 32
CLASSIFIER_LR = 2e-5
CLASSIFIER_MAX_LEN = 60   # Sesuaikan dengan MAX_LENGTH GAN jika perlu

# Berapa banyak data baru yang ingin kita buat?
# Dataloader GAN Anda (batch 64) punya 22 batch.
# Mari kita buat augmentasi sejumlah 22 batch (22 * 64 = 1408 sampel baru)
N_BATCHES_TO_GENERATE = len(dataloader)

In [ ]:
def generate_augmentation_data(generator_model, emb_layer, data_loader, voc_obj, n_batches):
    """
    Menjalankan cGAN untuk menghasilkan data augmentasi baru.
    Hanya menghasilkan data untuk kelas 'disaster' (target=1).
    """
    generator_model.eval()
    all_generated_texts = []
    
    print(f"Mulai menghasilkan {n_batches} batch data augmentasi...")
    
    # Ambil bobot embedding untuk decoding
    embedding_weights = emb_layer.weight.data
    
    data_iterator = iter(data_loader)
    for i in tqdm(range(n_batches)):
        try:
            # Ambil batch data asli (sebagai 'kondisi')
            real_indices = next(data_iterator).to(device)
        except StopIteration:
            # Ulangi dataloader jika n_batches > len(dataloader)
            data_iterator = iter(data_loader)
            real_indices = next(data_iterator).to(device)

        with torch.no_grad():
            real_embeddings = emb_layer(real_indices)
            batch_size = real_embeddings.size(0)
            
            # Buat noise
            noise = torch.randn(batch_size, NOISE_DIM, device=device)
            
            # Hasilkan embedding palsu (conditional)
            fake_embeddings = generator_model(real_embeddings, noise)
            
            # Decode kembali ke teks
            generated_texts = decode_embeddings(fake_embeddings, embedding_weights, voc_obj)
            all_generated_texts.extend(generated_texts)
            
    print(f"Selesai! Menghasilkan {len(all_generated_texts)} tweet augmentasi baru.")
    return all_generated_texts

In [ ]:
# 1. Jalankan fungsi augmentasi
# (Pastikan model 'generator', 'embedding_layer', 'dataloader', 'voc' ada di memori)
generated_tweets_text = generate_augmentation_data(
    generator, 
    embedding_layer, 
    dataloader, 
    voc, 
    N_BATCHES_TO_GENERATE
)

# 2. Buat DataFrame dari data augmentasi
# (Kita tahu ini semua adalah 'target=1' karena kita melatihnya)
df_new_data = pd.DataFrame({
    'text': generated_tweets_text,
    'target': 1
})

# 3. Muat data ASLI secara keseluruhan (termasuk target=0)
print(f"Memuat data asli dari {DATA_PATH}")
df_original = pd.read_csv(DATA_PATH)
df_original = df_original[['text', 'target']].dropna()

# 4. Buat DataFrame AUGMENTASI (Gabungan)
print("Membuat DataFrame augmentasi...")
df_augmented = pd.concat([df_original, df_new_data], ignore_index=True)

# 5. Lakukan Train/Validation Split untuk KEDUA skenario
print(f"Data Asli Total: {len(df_original)}")
print(f"Data Augmentasi Total: {len(df_augmented)}")

# Split A (Original)
train_df_orig, val_df_orig = train_test_split(
    df_original, 
    test_size=0.2, 
    random_state=42, 
    stratify=df_original['target']
)

# Split B (Augmented)
train_df_aug, val_df_aug = train_test_split(
    df_augmented, 
    test_size=0.2, 
    random_state=42, 
    stratify=df_augmented['target']
)

print(f"Split A: Train {len(train_df_orig)}, Val {len(val_df_orig)}")
print(f"Split B: Train {len(train_df_aug)}, Val {len(val_df_aug)}")

In [ ]:
# 1. Load Tokenizer
classifier_tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_MODEL_NAME)

# 2. Buat PyTorch Dataset untuk Classifier
class ClassifierDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.dataframe = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.texts = dataframe['text'].values
        self.labels = dataframe['target'].values

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        text = str(self.texts[index])
        label = int(self.labels[index])

        # Tokenisasi teks
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
def train_classifier_epoch(model, data_loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for batch in tqdm(data_loader, desc="Training Classifier"):
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        total_loss += loss.item()
        
        loss.backward()
        optimizer.step()
        scheduler.step()
        
    return total_loss / len(data_loader)

def eval_classifier(model, data_loader, device):
    model.eval()
    total_loss = 0
    all_preds = []   # Kita akan kumpulkan prediksi di sini
    all_labels = []  # Kita akan kumpulkan label asli di sini
    
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating Classifier"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            loss = outputs.loss
            total_loss += loss.item()
            
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)
            
            # Kumpulkan hasil dari setiap batch
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    # Hitung F1 Score akhir menggunakan sklearn
    final_f1 = f1_score(all_labels, all_preds, average='macro')
    return total_loss / len(data_loader), final_f1

def run_experiment(train_df, val_df, model_name, epochs, batch_size, lr, max_len):
    """Fungsi helper untuk menjalankan satu eksperimen penuh."""
    
    # 1. Inisialisasi Model
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
    
    # 2. Buat DataLoaders
    train_dataset = ClassifierDataset(train_df, classifier_tokenizer, max_len)
    val_dataset = ClassifierDataset(val_df, classifier_tokenizer, max_len)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    # 3. Optimizer & Scheduler
    optimizer = AdamW(model.parameters(), lr=lr)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
    
    best_f1 = 0.0
    for epoch in range(1, epochs + 1):
        print(f"\n--- Epoch {epoch}/{epochs} ---")
        
        train_loss = train_classifier_epoch(model, train_loader, optimizer, scheduler, device)
        val_loss, val_f1 = eval_classifier(model, val_loader, device)
        
        print(f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1 (Macro): {val_f1:.4f}")
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            # (Opsional: Anda bisa menyimpan model terbaik di sini)
            
    return best_f1

In [ ]:
print("="*50)
print("🚀 Menjalankan Eksperimen A: HANYA Data Asli...")
print("="*50)

f1_original = run_experiment(
    train_df_orig, 
    val_df_orig, 
    CLASSIFIER_MODEL_NAME, 
    CLASSIFIER_EPOCHS, 
    CLASSIFIER_BATCH_SIZE, 
    CLASSIFIER_LR, 
    CLASSIFIER_MAX_LEN
)

print("\n" + "="*50)
print("🚀 Menjalankan Eksperimen B: Data Asli + Augmentasi GAN...")
print("="*50)

f1_augmented = run_experiment(
    train_df_aug, 
    val_df_orig, 
    CLASSIFIER_MODEL_NAME, 
    CLASSIFIER_EPOCHS, 
    CLASSIFIER_BATCH_SIZE, 
    CLASSIFIER_LR, 
    CLASSIFIER_MAX_LEN
)


print("\n\n" + "="*50)
print("          HASIL AKHIR EKSPERIMEN          ")
print("="*50)
print(f"📈 F1 Score (Data Asli):      {f1_original:.4f}")
print(f"🚀 F1 Score (Data Augmentasi): {f1_augmented:.4f}")
print("="*50)

if f1_augmented > f1_original:
    print("\n🎉 SELAMAT! Model GAN Anda berhasil meningkatkan performa classifier!")
    print(f"Peningkatan F1 Score: +{f1_augmented - f1_original:.4f}")
else:
    print("\nModel GAN tidak meningkatkan performa.")
    print("Ini bisa terjadi jika data augmentasi tidak cukup beragam atau berkualitas tinggi.")
    print("Coba latih GAN lebih lama atau tuning hyperparameter GAN.")